# Credit Card Default (Fair DP-GANs) 

Author: Ilse Harmers \
Last modified: April 21, 2026

In [ ]:
# Importing libraries.
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
from snsynth import Synthesizer
import snsynth.transform as tf
import utils
import warnings
warnings.filterwarnings("ignore", message = r"Using", category = FutureWarning)
warnings.filterwarnings("ignore", message = r"invalid", category = RuntimeWarning)

In [ ]:
# Importing train set.
credit_train = pd.read_csv("./train-test-datasets/credit-card-default/credit_train.csv")

# Preprocessing the dataset for the Fair DP-GANs; the dataset Fair DP-GANs are expecting the first two columns in the original dataset to be the 
# sensitive and target attributes. 
cols = credit_train.columns.to_list()
cols.remove("SEX")
cols.remove("DEFAULT")
credit_train = credit_train[["SEX", "DEFAULT"] + cols]

print(credit_train.columns.to_list())
credit_train.describe()

In [ ]:
# Setting up preprocessor table transformer.

tt = tf.TableTransformer([
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # SEX
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # DEFAULT
    tf.ChainTransformer([
        tf.LogTransformer(),
        tf.MinMaxTransformer(lower = np.log(credit_train["LIMIT_BAL"].min()), 
                             upper = np.log(credit_train["LIMIT_BAL"].max()),   # 1 id with LIMIT_BAL = 1000000
                             negative = False) # LIMIT_BAL; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # EDUCATION
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # MARRIAGE
    tf.MinMaxTransformer(lower = credit_train["AGE"].min(), 
                         upper = credit_train["AGE"].max(),   # 1 id with AGE = 79
                         negative = False), # AGE; scaling to range (0, 1)
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # PAY_0
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # PAY_2
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # PAY_3
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # PAY_4
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # PAY_5
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # PAY_6
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = -1 * (np.log(abs(-166000.0) + 1)), 
                             upper = np.log(965000.0 + 1), negative = False) # BILL_AMT1; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = -1 * (np.log(abs(-70000.0) + 1)), 
                             upper = np.log(984000.0 + 1), negative = False) # BILL_AMT2; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = -1 * (np.log(abs(-157000.0) + 1)), 
                             upper = np.log(1664000.0 + 1), negative = False) # BILL_AMT3; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = -1 * (np.log(abs(-170000.0) + 1)), 
                             upper = np.log(892000.0 + 1), negative = False) # BILL_AMT4; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = -1 * (np.log(abs(-81000.0) + 1)), 
                             upper = np.log(927000.0 + 1), negative = False) # BILL_AMT5; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = -1 * (np.log(abs(-340000.0) + 1)), 
                             upper = np.log(962000.0 + 1), negative = False) # BILL_AMT6; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = 0, upper = np.log(874000.0 + 1), 
                             negative = False) # PAY_AMT1; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = 0, upper = np.log(1684000.0 + 1), 
                             negative = False) # PAY_AMT2; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = 0, upper = np.log(896000.0 + 1), 
                             negative = False) # PAY_AMT3; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = 0, upper = np.log(621000.0 + 1), 
                             negative = False) # PAY_AMT4; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = 0, upper = np.log(427000.0 + 1), 
                             negative = False) # PAY_AMT5; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = 0, upper = np.log(529000.0 + 1), 
                             negative = False) # PAY_AMT6; scaling to range (0, 1)
    ]),
])

In [ ]:
# Defining delta as the inverse of the size of the dataset: if a dataset has 2 * 10^4 rows, then delta = 10^(-5). 
delta = 10**np.floor(np.log10(1/credit_train.shape[0]))
print(delta)

# Defining beta1 in Adam optimizer.
beta1 = 0.9 

# Defining epsilon value.
epsi = 2

# Defining Fair DP-GAN: [dem, dis, dem-dis, clf_dem, clf_dis, clf_eqopp, clf_all].
fair_loss = "clf_eqopp"

In [ ]:
r = 1
while r < 6:
    
    print(f"Run {r}")

    # Synthesizing the dataset with one of the Fair DP-GANs.
    synth = Synthesizer.create('fairdpgan', s = "SEX", s_unpriv = 2, s_priv = 1, y = "DEFAULT", y_des = 1, fair_loss = fair_loss,
                               epsilon = epsi, delta = delta, beta1 = beta1, epoch_time = True, batch_size = 512, dataset = "Credit",
                               #sanity_check = True
                              )
    synth.fit(credit_train, transformer = tt, preprocessor_eps = 0.0)
        
    r += 1

In [ ]:
# Saving the results on runtime.
#time_per_run = pd.DataFrame({"time": [0.645514, 0.645694, 0.652845, 0.640464, 0.628679]}, index = [f"run{i}" for i in range(1, 6)])
#time_per_run = pd.DataFrame({"time": [0.651057, 0.665152, 0.662060, 0.655171, 0.663663]}, index = [f"run{i}" for i in range(1, 6)])
#time_per_run = pd.DataFrame({"time": [0.653267, 0.662744, 0.665553, 0.657313, 0.662754]}, index = [f"run{i}" for i in range(1, 6)])
#time_per_run = pd.DataFrame({"time": [7.924410, 7.934121, 7.938773, 7.930215, 7.924743]}, index = [f"run{i}" for i in range(1, 6)])
#time_per_run = pd.DataFrame({"time": [7.938468, 7.964409, 7.949084, 7.928797, 7.929485]}, index = [f"run{i}" for i in range(1, 6)])
#time_per_run = pd.DataFrame({"time": [8.099570, 8.069115, 8.085956, 8.076871, 8.076665]}, index = [f"run{i}" for i in range(1, 6)])
#time_per_run = pd.DataFrame({"time": [8.173326, 8.109728, 8.113232, 8.110512, 8.115883]}, index = [f"run{i}" for i in range(1, 6)])
time_per_run.to_csv(f"./Runtimes/credit_runtime_{fair_loss}.csv")
time_per_run

In [ ]:
# Computing average runtime across the runs.
print("Mean runtime [s/epoch]:", np.mean(time_per_run["time"]))
print("Standard deviation runtime [s/epoch]:", np.std(time_per_run["time"]))